In [1]:
import os

import numpy as np
import pandas as pd

from scripts.data_loader import load_dataframe
from scripts.data_processing import cluster_data_and_train_random_forest
from scripts.utils import extract_unique_npcis, RF_PARAM

# source files
BASE_DIR = "data/"
FULL_DATA_SET = "Campaign_data_NBIoT_1_2_3_4_5_6_interpolated_smoothed.mat"
filename = os.path.join(BASE_DIR, FULL_DATA_SET)

# Series of random seeds for reproducability
random_seeds = np.loadtxt('data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename)

Loaded dataframe from .h5 file: data/Campaign_data_NBIoT_1_2_3_4_5_6_interpolated_smoothed.mat_dataframe.h5


In [ ]:
from scripts.weighted_coverage import create_point_matrix, compute_weights, wknn_one


def process_single_test_point(single_tp, df_rp, unique_npcis, rf_param, k_max):
    # Ensure single_tp is a DataFrame with one row for consistency
    df_tp = pd.DataFrame([single_tp])

    # Create the point matrix for the reference points
    m_rfp, idx_rfp = create_point_matrix(df_rp, unique_npcis, rf_param)

    # Create the point matrix for the single test point
    m_tp, idx_tp = create_point_matrix(df_tp, unique_npcis, rf_param)

    # Compute the weights between the single test point and reference points
    W, idx_sort = compute_weights(m_rfp, idx_rfp, m_tp, idx_tp)

    #print(f"* MATRIX RP {m_rfp.shape} TP {m_tp.shape} W {W.shape}")

    # Do wKNN to estimate the position and error for the single test point
    TP_est_location, k_avg_error, selected_RPS = wknn_one(
        df_tp, df_rp, idx_sort, W, k_max
    )

    return TP_est_location, k_avg_error, selected_RPS, W

# Example usage:
# single_test_point = df_tp.iloc[0]  # Assuming df_tp is your test points DataFrame
# result = process_single_test_point(single_test_point, df_rp, unique_npcis, rf_param, k_max)


In [ ]:
rand = 42
#df = df.sample(20, random_state=rand)

In [ ]:
from itertools import chain
from scripts.utils import dataset_tp_rp_split
from scripts.weighted_coverage import process_test_points

unique_npcis = extract_unique_npcis(df, [1, 10, 88])
rf_param = RF_PARAM.NSINR
n_clusters = 2

print(df.index)
rf_model = cluster_data_and_train_random_forest(df, n_clusters, unique_npcis, rf_param, rand)

df_tp, df_rp = dataset_tp_rp_split(df, 0.3, rand)

print(f'TPS {df_tp.shape[0]} RPS {df_rp.shape[0]}')
print(df_tp['cluster'].value_counts())


def investiage_tp(idx: int):
    print(f"=====INVESTIGATING {idx} ======")
    tp = df_tp.loc[idx]

    print("\tTesting without clustering")
    print(f"\tReference points {df_rp.index.values}")
    TP_est_location, k_avg_error, selected_RPS, W = process_single_test_point(tp, df_rp, unique_npcis, rf_param, 2)

    print("\tTesting with clustering for TP")
    c = tp['cluster']
    cluster_rps = df_rp[df_rp['cluster'] == c]
    print(f"\tReference points for {c} -> {cluster_rps.index.values}")
    cTP_est_location, ck_avg_error, cW = process_single_test_point(tp, cluster_rps, unique_npcis,
                                                                   rf_param,
                                                                   2)
    return k_avg_error == ck_avg_error


# for i in range(5):
#     print(f'res {i}: {investiage_tp(i)}')


TP_est_location, errors_base, W = process_test_points(df_tp, df_rp, unique_npcis, rf_param, 2)

cluster_errors = []
for cluster, group in df_tp.groupby('cluster'):
    tps = group
    rps = df_rp[df_rp['cluster'] == cluster]

    _, errors_base, _ = process_test_points(df_tp, df_rp, unique_npcis, rf_param, 2)
    cluster_errors.append(errors_base)

all_errors_flat = list(chain.from_iterable(cluster_errors))

all_errors_flat = np.array(all_errors_flat).mean()

print(f'Error base mean {errors_base.mean()}')
print(f'Error cluster mean {all_errors_flat.mean()}')


In [2]:

from scripts.weighted_coverage import run_weighted_coverage

k_val_wknn = 2
unique_npcis = extract_unique_npcis(df, [1, 10, 88])
rf_param = RF_PARAM.NSINR

#df = df.sample(100)  # limit number of points for debugging

n_runs = 2
n_clusters = 4

cluster_range = range(0, n_clusters + 1)
errors_dict = {c: [] for c in cluster_range}
complexity_dict = {c: [] for c in cluster_range}
runtime_dict = {c: [] for c in cluster_range}

for c in cluster_range:
    for i in range(n_runs):
        print(f"\r🔄 Running for {c} clusters ({i + 1}/{n_runs} runs)", end="")
        random = random_seeds[i]

        _, errors, complexity, runtime = run_weighted_coverage(
            df, rf_param, k_val_wknn, unique_npcis, random, c
        )
        mean_errors = errors.mean()
        errors_dict[c].append(mean_errors)
        complexity_dict[c].append(complexity)
        runtime_dict[c].append(runtime)
    print(f"\r✅ {c} clusters completed                                ")

errors_df = pd.DataFrame(errors_dict)
complexity_df = pd.DataFrame(complexity_dict)
runtime_df = pd.DataFrame(runtime_dict)


✅ 0 clusters completed                                
✅ 1 clusters completed                                
✅ 2 clusters completed                                
✅ 3 clusters completed                                
✅ 4 clusters completed                                


In [5]:
runtime_df

,0,1,2,3,4
0,1.314929,1.851315,1.670669,1.499954,1.542238
1,1.197969,1.781937,1.675690,1.664267,1.554797


In [ ]:
from scripts.plotting import make_boxplot

base_error = cluster_df['0'].mean()
title_string = f"wKNN with K-Means & Random Forest\n{rf_param.value} ✴︎ wKNN K={k_val_wknn} ✴︎ Operators: {', '.join(['1', '10', '88'])} ✴︎ {n_runs} runs,"

total_df = pd.concat([cluster_df, cluster_df2], axis=1)

make_boxplot(
    df=total_df,
    title=f'Min average error: {title_string}',
    x_label='Number of clusters for K-Means',
    y_label='Error (m)',
    color='mediumseagreen',
    baseline=base_error,
)
